# Introduction 3: text embeddings with model2vec

`model2vec` turns a piece of text into a vector with a *static* embedding model: every token has one fixed vector and
a sentence vector is their average. We use it in preprocessing to find which parts of the graph schema a question is
about (see `notebooks/preprocessing/01_t2css_pipeline.ipynb`). It needs no GPU and no API key.

In [1]:
import time, warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np
import pandas as pd
import model2vec
from model2vec import StaticModel

model = StaticModel.from_pretrained("minishlab/potion-base-8M")
print("model2vec", model2vec.__version__, "| vocabulary x dimensions:", model.embedding.shape)

model2vec 0.10.0 | vocabulary x dimensions: (29528, 256)


## Encode a few texts

`encode` takes a list of strings and returns one row per string.

In [2]:
texts = ["Which compounds treat multiple sclerosis?",
         "compound treats disease",
         "compound binds gene",
         "gene participates in pathway"]
vectors = np.asarray(model.encode(texts))
print(vectors.shape, vectors.dtype, "| vector lengths:", np.round(np.linalg.norm(vectors, axis=1), 3))

(4, 256) float32 | vector lengths: [1. 1. 1. 1.]


## Cosine similarity

The vectors have length 1, so the cosine similarity of two texts is simply their dot product. The question is closest
to the statement that says the same thing.

In [3]:
pd.DataFrame(vectors @ vectors.T, index=texts, columns=[f"[{i}]" for i in range(len(texts))]).round(3)

,[0],[1],[2],[3]
Which compounds treat multiple sclerosis?,1.000,0.617,0.344,0.052
compound treats disease,0.617,1.000,0.449,0.051
compound binds gene,0.344,0.449,1.000,0.341
gene participates in pathway,0.052,0.051,0.341,1.000


## What "static" means

Tokenise a text, look up the stored vector of each token, average them and normalise: the result is the same vector
`encode` returns. There is no neural network pass at inference time.

In [4]:
text = "compound treats disease"
enc = model.tokenizer.encode(text, add_special_tokens=False)
mean = model.embedding[enc.ids].mean(axis=0)
mean = mean / np.linalg.norm(mean)
print("tokens:", enc.tokens)
print("mean of token vectors equals encode():", bool(np.allclose(mean, np.asarray(model.encode([text]))[0], atol=1e-5)))

tokens: ['compound', 'treats', 'disease']
mean of token vectors equals encode(): True


## Deterministic and fast

The same text always gives the same vector, which matters for a reproducibility study.

In [5]:
again = np.asarray(model.encode(texts))
print("identical on a second call:", bool(np.array_equal(vectors, again)))
batch = [f"compound number {i} treats disease number {i}" for i in range(1000)]
t0 = time.perf_counter(); model.encode(batch); dt = time.perf_counter() - t0
print("1000 short texts encoded on CPU in under a second:", dt < 1.0)

identical on a second call: True
1000 short texts encoded on CPU in under a second: True


## Nearest neighbours

Ranking candidates by similarity to a query is one matrix product and a sort.

In [6]:
candidates = ["compound treats disease", "compound palliates disease", "compound causes side effect",
              "disease presents symptom", "gene interacts with gene"]
query = np.asarray(model.encode(["Which drugs relieve the symptoms of asthma?"]))[0]
scores = np.asarray(model.encode(candidates)) @ query
pd.DataFrame({"candidate": candidates, "similarity": scores.round(3)}).sort_values("similarity", ascending=False, ignore_index=True)

,candidate,similarity
0,compound treats disease,0.400
1,compound causes side effect,0.347
2,disease presents symptom,0.296
3,compound palliates disease,0.175
4,gene interacts with gene,0.005


## Takeaways

- One call, `model.encode(list_of_texts)`, returns unit-length vectors; similarity is a dot product.
- A static model is an average of token vectors, so it is cheap and exactly repeatable.
- It matches words, not logic: it cannot tell "treats" from "does not treat". The preprocessing notebook measures what
  that means for our questions.